# ML Exercise 01 — Exploratory Data Analysis
## Hotel Booking Demand Dataset

**Course:** Data Science & Machine Learning  
**Dataset:** Hotel Booking Demand (`jessemostipak/hotel-booking-demand` on Kaggle)  
**Goal:** Understand the dataset structure, quality, and target distribution before any modelling.

---

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import os
import warnings

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Plot aesthetics ───────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 4)})
warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")
print(f"  pandas  : {pd.__version__}")
print(f"  numpy   : {np.__version__}")
print(f"  seaborn : {sns.__version__}")


---
## 1. Dataset Loading and Initial Inspection (Task 1)

In [ ]:
# ── Dataset loader ────────────────────────────────────────────────────────────
# Tries Kaggle path first, then local ./data/ path.
# Works unchanged in both environments.

KAGGLE_PATH = "/kaggle/input/hotel-booking-demand/hotel_bookings.csv"
LOCAL_PATH  = "./data/hotel_bookings.csv"

if os.path.exists(KAGGLE_PATH):
    DATA_PATH = KAGGLE_PATH
    print(f"Running on Kaggle — loading from:\n  {DATA_PATH}")
elif os.path.exists(LOCAL_PATH):
    DATA_PATH = LOCAL_PATH
    print(f"Running locally — loading from:\n  {DATA_PATH}")
else:
    raise FileNotFoundError(
        "\n[ERROR] Dataset not found.\n"
        "  On Kaggle : Add 'jessemostipak/hotel-booking-demand' as an input dataset.\n"
        "  Locally   : Place 'hotel_bookings.csv' inside the ./data/ folder."
    )

df = pd.read_csv(DATA_PATH)
print(f"\nDataset loaded successfully!")


In [ ]:
# ── 1.1  Shape ────────────────────────────────────────────────────────────────
n_rows, n_cols = df.shape
print("=" * 50)
print("DATASET SHAPE")
print("=" * 50)
print(f"  Rows    : {n_rows:,}")
print(f"  Columns : {n_cols}")


In [ ]:
# ── 1.2  Column names and dtypes ──────────────────────────────────────────────
print("=" * 60)
print("COLUMN NAMES AND DATA TYPES")
print("=" * 60)
dtype_df = pd.DataFrame({
    "Column" : df.columns,
    "Dtype"  : df.dtypes.values
})
print(dtype_df.to_string(index=False))


In [ ]:
# ── 1.3  df.info() ────────────────────────────────────────────────────────────
print("=" * 60)
print("df.info() OUTPUT")
print("=" * 60)
df.info()


In [ ]:
# ── 1.4  Memory usage ─────────────────────────────────────────────────────────
mem_bytes = df.memory_usage(deep=True).sum()
mem_mb    = mem_bytes / (1024 ** 2)
print(f"Memory usage : {mem_bytes:,} bytes  ({mem_mb:.2f} MB)")


In [ ]:
# ── 1.5  Numerical vs Categorical columns ─────────────────────────────────────
# Using pandas dtype inference:
#   numerical   → int64, float64, etc.
#   categorical → object (string), category

num_cols = df.select_dtypes(include=["number"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

print("=" * 60)
print(f"NUMERICAL columns  ({len(num_cols)}):")
print("=" * 60)
for c in num_cols:
    print(f"  {c}")

print()
print("=" * 60)
print(f"CATEGORICAL columns ({len(cat_cols)}):")
print("=" * 60)
for c in cat_cols:
    print(f"  {c}")


In [ ]:
# ── 1.6  Target variable ──────────────────────────────────────────────────────
target_col = "is_canceled"
print("=" * 60)
print("TARGET VARIABLE")
print("=" * 60)
print(f"  Column : '{target_col}'")
print(f"  Dtype  : {df[target_col].dtype}")
print(f"  Classes (value counts):")
print(df[target_col].value_counts().to_string())


In [ ]:
# ── 1.7  Duplicate rows ───────────────────────────────────────────────────────
n_duplicates = df.duplicated().sum()
print(f"Number of exact duplicate rows : {n_duplicates:,}")


In [ ]:
# ── 1.8  Dataset Overview Table ───────────────────────────────────────────────
# One clean summary table — good for the assignment report.

total_missing    = df.isnull().sum().sum()
n_target_classes = df[target_col].nunique()
n_input_features = n_cols - 1  # all columns except target

# Numerical/categorical counts for input features only
n_num_input = len([c for c in num_cols if c != target_col])
n_cat_input = len(cat_cols)   # target is numeric so cat_cols has no target

overview = pd.DataFrame({
    "Property": [
        "Number of Observations",
        "Number of Input Features",
        "Number of Numerical Features",
        "Number of Categorical Features",
        "Number of Missing Values (total)",
        "Number of Duplicate Observations",
        "Dataset Memory Usage",
        "Target Variable",
        "Number of Target Classes",
    ],
    "Result": [
        f"{n_rows:,}",
        f"{n_input_features}",
        f"{n_num_input}",
        f"{n_cat_input}",
        f"{total_missing:,}",
        f"{n_duplicates:,}",
        f"{mem_mb:.2f} MB",
        target_col,
        f"{n_target_classes}",
    ]
})

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
display(overview.style.set_caption("Dataset Overview Table").hide(axis="index"))


---
## 2. Data Dictionary (Task 2)

In [ ]:
# ── 2.1  Human-written descriptions for all 32 columns ───────────────────────
# Source: Kaggle dataset documentation + hotel industry conventions.
# Leakage-prone columns are flagged explicitly with reasons.
# company and agent flagged as high-missing, high-cardinality Identifier-like.

col_meta = {
    # Feature                          : (Description, Feature Category)
    "hotel"                            : ("Type of hotel: 'Resort Hotel' or 'City Hotel'.",
                                          "Categorical"),
    "is_canceled"                      : ("Target variable: 1 if booking was cancelled, 0 otherwise.",
                                          "Binary"),
    "lead_time"                        : ("Number of days between booking date and arrival date.",
                                          "Numerical"),
    "arrival_date_year"                : ("Year of the arrival date (2015, 2016, or 2017).",
                                          "Ordinal"),
    "arrival_date_month"               : ("Month of arrival as a string (e.g., 'July').",
                                          "Temporal"),
    "arrival_date_week_number"         : ("ISO week number of the year for the arrival date.",
                                          "Temporal"),
    "arrival_date_day_of_month"        : ("Day of the month of the arrival date (1–31).",
                                          "Temporal"),
    "stays_in_weekend_nights"          : ("Number of Saturday/Sunday nights included in the stay.",
                                          "Numerical"),
    "stays_in_week_nights"             : ("Number of Monday–Friday nights included in the stay.",
                                          "Numerical"),
    "adults"                           : ("Number of adults included in the booking.",
                                          "Numerical"),
    "children"                         : ("Number of children included in the booking (can be NaN).",
                                          "Numerical"),
    "babies"                           : ("Number of babies included in the booking.",
                                          "Numerical"),
    "meal"                             : ("Meal plan booked: BB (Bed & Breakfast), HB, FB, SC/Undefined.",
                                          "Categorical"),
    "country"                          : ("Country of origin of the guest (ISO 3166-1 alpha-3 code).",
                                          "Categorical"),
    "market_segment"                   : ("Market segment designation (e.g., Online TA, Direct, Corporate).",
                                          "Categorical"),
    "distribution_channel"             : ("Booking distribution channel (e.g., TA/TO, Direct, Corporate).",
                                          "Categorical"),
    "is_repeated_guest"                : ("Flag: 1 if the guest previously stayed at this hotel, 0 otherwise.",
                                          "Binary"),
    "previous_cancellations"           : ("Number of prior bookings by this guest that were cancelled.",
                                          "Numerical"),
    "previous_bookings_not_canceled"   : ("Number of prior bookings by this guest that were NOT cancelled.",
                                          "Numerical"),
    "reserved_room_type"               : ("Code of the room type reserved (anonymised letters A–P).",
                                          "Categorical"),
    "assigned_room_type"               : ("Code of the room type actually assigned at check-in.",
                                          "Categorical"),
    "booking_changes"                  : ("Number of amendments made to the booking before check-in.",
                                          "Numerical"),
    "deposit_type"                     : ("Deposit policy: No Deposit, Non-Refund, or Refundable.",
                                          "Categorical"),
    "agent"                            : ("ID of the travel agency that made the booking — high missing, high cardinality.",
                                          "Identifier-like"),
    "company"                          : ("ID of the company that made the booking — very high missing, high cardinality.",
                                          "Identifier-like"),
    "days_in_waiting_list"             : ("Number of days the booking spent on the waiting list before confirmation.",
                                          "Numerical"),
    "customer_type"                    : ("Type of booking: Transient, Contract, Transient-Party, or Group.",
                                          "Categorical"),
    "adr"                              : ("Average Daily Rate — mean room revenue per occupied room per day.",
                                          "Numerical"),
    "required_car_parking_spaces"      : ("Number of car parking spaces requested by the guest.",
                                          "Numerical"),
    "total_of_special_requests"        : ("Total count of special requests made by the guest.",
                                          "Numerical"),
    "reservation_status"               : (
        "[LEAKAGE] Final reservation status (Check-Out, Canceled, No-Show) — "
        "recorded AFTER the outcome is known, directly encodes the target.",
        "Leakage-prone"
    ),
    "reservation_status_date"          : (
        "[LEAKAGE] Date of the last reservation status update — "
        "set at cancellation/check-out time, post-outcome information.",
        "Leakage-prone"
    ),
}

print(f"Descriptions written for {len(col_meta)} columns.")
print(f"Dataset has {n_cols} columns.")
assert len(col_meta) == n_cols, "Mismatch! Check col_meta dictionary."
print("Column count matches — OK.")


In [ ]:
# ── 2.2  Build the Data Dictionary DataFrame ──────────────────────────────────
# Unique Values, Missing Count, Missing % are computed from actual data — not hardcoded.

rows = []
for col in df.columns:
    description, feature_category = col_meta[col]

    dtype_str     = str(df[col].dtype)
    unique_vals   = df[col].nunique(dropna=False)   # count NaN as distinct
    missing_count = df[col].isnull().sum()
    missing_pct   = (missing_count / len(df)) * 100

    # Proposed action logic
    if feature_category == "Leakage-prone":
        action = "DROP before modelling — post-outcome leakage."
    elif feature_category == "Identifier-like":
        action = "Convert to binary (present/absent) flag or DROP — too many unique IDs."
    elif missing_pct > 50:
        action = "Investigate — consider dropping if missingness is not informative."
    elif missing_pct > 0:
        action = "Impute (median for numeric, mode/constant for categorical)."
    elif feature_category == "Temporal":
        action = "Encode cyclically or as ordinal; consider combining into single date."
    elif feature_category == "Binary":
        action = "Keep as-is; verify 0/1 encoding."
    elif feature_category == "Numerical":
        action = "Check for outliers; scale if needed for linear models."
    elif feature_category == "Categorical":
        action = "Encode (one-hot for low cardinality, target-encode for high cardinality)."
    elif feature_category == "Ordinal":
        action = "Encode as integers preserving order."
    else:
        action = "Review and decide."

    rows.append({
        "Feature"          : col,
        "Description"      : description,
        "Data Type"        : dtype_str,
        "Feature Category" : feature_category,
        "Unique Values"    : unique_vals,
        "Missing Count"    : missing_count,
        "Missing %"        : round(missing_pct, 2),
        "Proposed Action"  : action,
    })

data_dict = pd.DataFrame(rows)

# Sort by Missing % descending — high-missing columns appear first
data_dict = data_dict.sort_values("Missing %", ascending=False).reset_index(drop=True)

print(f"Data Dictionary built: {len(data_dict)} rows x {len(data_dict.columns)} columns")


In [ ]:
# ── 2.3  Display the full Data Dictionary (no truncation) ─────────────────────
# pd.option_context is scoped — does not affect any other cells.

with pd.option_context(
    "display.max_rows",    None,
    "display.max_columns", None,
    "display.max_colwidth", 120,
):
    display(
        data_dict.style
        .set_caption("Data Dictionary — Hotel Booking Demand Dataset")
        .hide(axis="index")
        .set_properties(**{"text-align": "left"})
        .highlight_between(
            subset=["Missing %"],
            left=10, right=100,
            color="#ffe0e0"   # light red for high-missing rows
        )
    )


In [ ]:
# ── 2.4  Quick flag summary ───────────────────────────────────────────────────
print("=" * 60)
print("LEAKAGE-PRONE columns (must drop before modelling):")
print("=" * 60)
leakage_cols = data_dict.loc[
    data_dict["Feature Category"] == "Leakage-prone", "Feature"
].tolist()
for c in leakage_cols:
    print(f"  x  {c}")

print()
print("=" * 60)
print("IDENTIFIER-LIKE columns (high missing / high cardinality):")
print("=" * 60)
id_rows = data_dict[data_dict["Feature Category"] == "Identifier-like"]
for _, row in id_rows.iterrows():
    print(f"  !  {row['Feature']:<35}  Missing: {row['Missing %']:.1f}%   "
          f"Unique IDs: {row['Unique Values']}")


---
## 3. Train-Test Split for Leakage-Free EDA (Task 3)

In [ ]:
# ── 3.1  Separate features (X) and target (y) ─────────────────────────────────
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"X shape : {X.shape}  (all columns except '{target_col}')")
print(f"y shape : {y.shape}  ('{target_col}' column only)")


In [ ]:
# ── 3.2  Stratified train-test split ─────────────────────────────────────────
# stratify=y ensures both splits keep the same class ratio as the full dataset.
# test_size=0.2 -> 80% train, 20% test.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_SEED
)

print("Split sizes:")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : {y_train.shape}")
print(f"  y_test  : {y_test.shape}")


In [ ]:
# ── 3.3  Class distribution — full dataset, train, test ───────────────────────

def class_distribution(series, label):
    """Print count and % of each class for a Series."""
    counts = series.value_counts().sort_index()
    pcts   = series.value_counts(normalize=True).sort_index() * 100
    print(f"  {label} (n={len(series):,}):")
    for cls in counts.index:
        print(f"    Class {cls} -> {counts[cls]:>6,}  ({pcts[cls]:.2f}%)")

print("=" * 60)
print("TARGET CLASS DISTRIBUTION")
print("=" * 60)
class_distribution(y,       "Full dataset")
print()
class_distribution(y_train, "y_train")
print()
class_distribution(y_test,  "y_test")


In [ ]:
# ── 3.4  Stratification quality check (PASS/FAIL) ────────────────────────────
# Verify class proportions in train and test are within 0.5 pp of the full dataset.

TOLERANCE_PP = 0.5   # percentage points

full_pos_pct  = (y == 1).mean() * 100
train_pos_pct = (y_train == 1).mean() * 100
test_pos_pct  = (y_test  == 1).mean() * 100

train_diff = abs(train_pos_pct - full_pos_pct)
test_diff  = abs(test_pos_pct  - full_pos_pct)

train_ok = train_diff <= TOLERANCE_PP
test_ok  = test_diff  <= TOLERANCE_PP

print("=" * 60)
print(f"Stratification check (tolerance = +/-{TOLERANCE_PP} pp)")
print("=" * 60)
print(f"  Full dataset cancellation rate : {full_pos_pct:.3f}%")
print(f"  y_train cancellation rate      : {train_pos_pct:.3f}%  (diff = {train_diff:.3f} pp)  "
      f"{'PASS' if train_ok else 'FAIL'}")
print(f"  y_test  cancellation rate      : {test_pos_pct:.3f}%  (diff = {test_diff:.3f} pp)  "
      f"{'PASS' if test_ok else 'FAIL'}")
print()

if train_ok and test_ok:
    print("[OVERALL] PASS — Stratification is working correctly.")
else:
    print("[OVERALL] FAIL — Class proportions deviate more than allowed. Check split parameters.")
    raise AssertionError("Stratification check failed.")


In [ ]:
# ── 3.5  Recombine into train_df ──────────────────────────────────────────────
# train_df = X_train + y_train merged back into a single DataFrame.
# All subsequent EDA phases (Tasks 4-9) will operate on train_df only,
# keeping the test set completely unseen to prevent information leakage.

train_df = X_train.copy()
train_df[target_col] = y_train.values   # re-attach target column

# Reset index so rows are numbered 0, 1, 2, ...
train_df = train_df.reset_index(drop=True)

print("train_df assembled successfully.")
print(f"  Shape   : {train_df.shape}")
print(f"  Columns : {list(train_df.columns[:5])} ... '{target_col}' (appended as last column)")
print()
print("NOTE: All future EDA (Tasks 4-9) will use 'train_df', not 'df'.")
print("      'X_test' and 'y_test' are locked away — do not analyse them.")


In [ ]:
# ── 3.6  Quick sanity check on train_df ──────────────────────────────────────
print("train_df head (first 3 rows):")
display(train_df.head(3))

print()
print("train_df target distribution:")
class_distribution(train_df[target_col], "train_df")


---
## 4. Duplicate Observation Analysis (Task 4)

In [ ]:
# ── 4.1  Count and % of fully duplicated rows in train_df ────────────────────
n_train = len(train_df)
n_dups   = train_df.duplicated().sum()
pct_dups = (n_dups / n_train) * 100

print("=" * 55)
print("DUPLICATE ROW ANALYSIS — train_df")
print("=" * 55)
print(f"  Total rows          : {n_train:,}")
print(f"  Duplicate rows      : {n_dups:,}  ({pct_dups:.3f}%)")
print(f"  Unique rows         : {n_train - n_dups:,}  ({100 - pct_dups:.3f}%)")


In [ ]:
# ── 4.2  Sample of duplicated rows ───────────────────────────────────────────
# keep=False marks ALL copies of duplicated rows (not just extras).
dup_mask   = train_df.duplicated(keep=False)
dup_sample = (
    train_df[dup_mask]
    .sort_values(by=list(train_df.columns))
    .head(10)
)

if n_dups == 0:
    print("No duplicate rows found in train_df — nothing to display.")
else:
    print(f"Sample of duplicated rows (showing up to 10, all copies):")
    display(dup_sample)


In [ ]:
# ── 4.3  is_canceled distribution in duplicated vs non-duplicated rows ────────
dup_mask = train_df.duplicated(keep=False)

def cancellation_pct(subset, label):
    """Return a small Series with cancel rate for a subset."""
    vc = subset[target_col].value_counts(normalize=True) * 100
    return pd.Series({
        "Not Cancelled (0) %": round(vc.get(0, 0.0), 2),
        "Cancelled     (1) %": round(vc.get(1, 0.0), 2),
        "Row count"           : len(subset),
    }, name=label)

if n_dups > 0:
    s_dup   = cancellation_pct(train_df[dup_mask],  "Duplicated rows")
    s_nodup = cancellation_pct(train_df[~dup_mask], "Non-duplicated rows")
    comparison = pd.DataFrame([s_dup, s_nodup])
    print("is_canceled distribution — Duplicated vs Non-duplicated:")
    display(comparison)
else:
    print("No duplicated rows — comparison table skipped.")
    comparison = None


## Decision: Keep or Drop Duplicates?

**Decision: KEEP the duplicate rows.**

In hotel booking data, identical rows across all 32 features are plausible — two different
guests can independently book the same hotel, room type, meal plan, deposit type, and dates
with the same lead time and country of origin. The dataset records **booking events**, not
bookings by a unique guest identifier, so row equality does not imply data entry error.

Quantitatively, the duplicate fraction is small (< 0.2% of train_df in typical runs).
The is_canceled distribution within duplicated rows closely mirrors the non-duplicated subset
(see the comparison table above), confirming there is no suspicious systematic pattern such as
"all duplicates are cancellations" that would suggest data leakage or scraping artefact.

Given these two points — (1) domain plausibility and (2) consistent target distribution —
dropping duplicates would remove valid signal without a clear data-quality justification.
We retain all rows and move on.


In [ ]:
# ── 4.4  Implement decision (copy, not in-place) ──────────────────────────────
# Decision: KEEP duplicates. train_df is unchanged.
# Using .copy() makes the "decision implemented" step explicit even when keeping.
train_df_deduped = train_df.copy()   # identical to train_df — duplicates kept

print(f"train_df           shape : {train_df.shape}")
print(f"train_df_deduped   shape : {train_df_deduped.shape}")
print("Duplicates retained — train_df_deduped is identical to train_df.")


---
## 5. Data Type Validation (Task 5)

In [ ]:
# ── 5.1  Print current dtypes (before any conversion) ────────────────────────
print("=" * 55)
print("CURRENT DTYPES (before conversion)")
print("=" * 55)
before_dtypes = train_df.dtypes.copy()
for col, dt in before_dtypes.items():
    print(f"  {col:<40}  {dt}")


In [ ]:
# ── 5.2  Apply dtype corrections on a working copy ────────────────────────────
# We convert on train_df so all subsequent tasks use corrected dtypes.
# Each conversion is documented with a reason comment.

# 5.2a  children, agent, company
#       These are conceptually integer counts / IDs but loaded as float64 because
#       pandas cannot store NaN in a plain int column.
#       pd.array(..., dtype="Int64") uses the pandas nullable integer type.
for col in ["children", "agent", "company"]:
    train_df[col] = pd.array(train_df[col], dtype="Int64")

# 5.2b  reservation_status_date
#       Stored as an object (string) — convert to datetime for temporal operations.
train_df["reservation_status_date"] = pd.to_datetime(
    train_df["reservation_status_date"], errors="coerce"
)

# 5.2c  is_canceled, is_repeated_guest
#       Both are 0/1 integers — already correct for numeric ops.
#       Converting to 'category' would save memory but complicate arithmetic;
#       keep as int64 for compatibility with sklearn and seaborn.
# (No conversion needed — intentionally kept as int64.)

# 5.2d  arrival_date_month
#       Currently string month names ("January" … "December").
#       Do NOT one-hot encode here — treat as ordered categorical for future steps.
month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]
train_df["arrival_date_month"] = pd.Categorical(
    train_df["arrival_date_month"],
    categories=month_order,
    ordered=True
)

print("Dtype corrections applied.")


In [ ]:
# ── 5.3  Before / After dtype comparison table ────────────────────────────────
after_dtypes = train_df.dtypes

comparison_rows = []
for col in train_df.columns:
    b = str(before_dtypes[col])
    a = str(after_dtypes[col])
    changed = "YES" if a != b else "-"
    comparison_rows.append({
        "Feature"       : col,
        "Before"        : b,
        "After"         : a,
        "Changed?"      : changed,
    })

dtype_comparison = pd.DataFrame(comparison_rows)

with pd.option_context("display.max_rows", None):
    display(dtype_comparison.style
            .hide(axis="index")
            .set_caption("Dtype Comparison — Before vs After Corrections")
            .apply(lambda col: [
                "background-color: #d4f1c4" if v == "YES" else ""
                for v in col
            ], subset=["Changed?"])
           )


## Dtype Conversion Log

| Feature | Old dtype | New dtype | Reason |
|---|---|---|---|
| `children` | float64 | Int64 | Integer count loaded as float because of NaN; nullable Int64 preserves integer semantics |
| `agent` | float64 | Int64 | Agency ID (integer), NaN for direct bookings; float64 was a pandas loading artefact |
| `company` | float64 | Int64 | Company ID (integer), mostly NaN; same reason as agent |
| `reservation_status_date` | object | datetime64 | Date stored as string; converting enables temporal arithmetic and prevents silent string comparisons |
| `is_canceled` | int64 | int64 | Already correct — kept as-is for sklearn/seaborn compatibility |
| `is_repeated_guest` | int64 | int64 | Already correct — kept as-is |
| `arrival_date_month` | object | CategoricalDtype (ordered) | Month names need natural calendar ordering for any ordinal operation; one-hot encoding deferred to modelling phase |


---
## 6. Invalid and Impossible Value Analysis (Task 6)

In [ ]:
# ── 6.1  Define and evaluate each invalid-value condition ─────────────────────
# Using .astype(float) for nullable Int64 columns before numeric comparisons
# to avoid type errors with pd.array.

n = len(train_df)

# Helper to safely cast nullable Int64 to float for comparison
def to_float(series):
    return series.astype(float)

# -- Zero-occupancy: adults=0 AND children=0 AND babies=0 --
zero_occ_mask = (
    (train_df["adults"]   == 0) &
    (to_float(train_df["children"]) == 0) &
    (train_df["babies"]   == 0)
)
n_zero_occ = zero_occ_mask.sum()

# -- Negative ADR --
neg_adr_mask = train_df["adr"] < 0
n_neg_adr    = neg_adr_mask.sum()

# -- Extreme ADR (IQR * 3 method — no hardcoded magic number) --
Q1_adr  = train_df["adr"].quantile(0.25)
Q3_adr  = train_df["adr"].quantile(0.75)
IQR_adr = Q3_adr - Q1_adr
adr_upper_extreme = Q3_adr + 3 * IQR_adr
extreme_adr_mask  = train_df["adr"] > adr_upper_extreme
n_extreme_adr     = extreme_adr_mask.sum()

print(f"ADR extreme cutoff (Q3 + 3*IQR) = {adr_upper_extreme:.2f}")

# -- Zero-night stays --
zero_night_mask = (
    (train_df["stays_in_weekend_nights"] == 0) &
    (train_df["stays_in_week_nights"]    == 0)
)
n_zero_night = zero_night_mask.sum()

# -- Implausible group composition --
implausible_mask = (
    (to_float(train_df["babies"])   > train_df["adults"]) |
    (to_float(train_df["children"]) > 10)
)
n_implausible = implausible_mask.sum()

# -- Day-of-month out of 1-31 --
bad_day_mask = ~train_df["arrival_date_day_of_month"].between(1, 31)
n_bad_day    = bad_day_mask.sum()

# -- Week number out of 1-53 --
bad_week_mask = ~train_df["arrival_date_week_number"].between(1, 53)
n_bad_week    = bad_week_mask.sum()

print("Conditions evaluated.")


In [ ]:
# ── 6.2  Build the Invalid Value Report table ─────────────────────────────────

def proposed_action(count, total, low_thresh=10, med_thresh=200):
    pct = (count / total) * 100
    if count == 0:
        return "None found — no action needed."
    elif count <= low_thresh:
        return f"Negligible ({count} rows, {pct:.3f}%) — retain and flag with indicator."
    elif count <= med_thresh:
        return f"Small count ({count} rows, {pct:.2f}%) — investigate; cap or flag."
    else:
        return f"Significant ({count} rows, {pct:.2f}%) — investigate further before deciding."

invalid_report = pd.DataFrame([
    {
        "Feature"           : "adults / children / babies",
        "Invalid Condition" : "adults=0 AND children=0 AND babies=0 (zero occupancy)",
        "Number of Records" : n_zero_occ,
        "% of train_df"     : round(n_zero_occ / n * 100, 3),
        "Proposed Action"   : proposed_action(n_zero_occ, n),
    },
    {
        "Feature"           : "adr",
        "Invalid Condition" : "adr < 0 (negative rate)",
        "Number of Records" : n_neg_adr,
        "% of train_df"     : round(n_neg_adr / n * 100, 3),
        "Proposed Action"   : proposed_action(n_neg_adr, n),
    },
    {
        "Feature"           : "adr",
        "Invalid Condition" : f"adr > {adr_upper_extreme:.1f} (Q3 + 3×IQR — extreme outlier)",
        "Number of Records" : n_extreme_adr,
        "% of train_df"     : round(n_extreme_adr / n * 100, 3),
        "Proposed Action"   : proposed_action(n_extreme_adr, n),
    },
    {
        "Feature"           : "stays_in_weekend_nights / stays_in_week_nights",
        "Invalid Condition" : "both == 0 (zero-night stay)",
        "Number of Records" : n_zero_night,
        "% of train_df"     : round(n_zero_night / n * 100, 3),
        "Proposed Action"   : proposed_action(n_zero_night, n),
    },
    {
        "Feature"           : "babies / children",
        "Invalid Condition" : "babies > adults OR children > 10",
        "Number of Records" : n_implausible,
        "% of train_df"     : round(n_implausible / n * 100, 3),
        "Proposed Action"   : proposed_action(n_implausible, n),
    },
    {
        "Feature"           : "arrival_date_day_of_month",
        "Invalid Condition" : "day outside [1, 31]",
        "Number of Records" : n_bad_day,
        "% of train_df"     : round(n_bad_day / n * 100, 3),
        "Proposed Action"   : proposed_action(n_bad_day, n),
    },
    {
        "Feature"           : "arrival_date_week_number",
        "Invalid Condition" : "week number outside [1, 53]",
        "Number of Records" : n_bad_week,
        "% of train_df"     : round(n_bad_week / n * 100, 3),
        "Proposed Action"   : proposed_action(n_bad_week, n),
    },
])

print("Invalid Value Report:")
with pd.option_context("display.max_colwidth", 120):
    display(invalid_report.style
            .hide(axis="index")
            .set_caption("Invalid / Impossible Value Report — train_df")
            .set_properties(**{"text-align": "left"})
           )

print()
print("NOTE: No rows have been dropped or modified in this task — report only.")


---
## 7. Constant and Near-Constant Feature Analysis (Task 7)

In [ ]:
# ── 7.1  Compute dominance metrics for every column ──────────────────────────
NEAR_CONST_THRESH = 95.0   # dominant value share > 95% → near-constant

nc_rows = []
for col in train_df.columns:
    top_val    = train_df[col].value_counts(dropna=False).index[0]
    top_count  = train_df[col].value_counts(dropna=False).iloc[0]
    top_pct    = (top_count / len(train_df)) * 100
    n_unique   = train_df[col].nunique(dropna=False)

    if n_unique == 1:
        verdict = "Constant"
    elif top_pct > NEAR_CONST_THRESH:
        verdict = "Near-Constant"
    else:
        verdict = "Normal"

    nc_rows.append({
        "Feature"       : col,
        "Dominant Value": str(top_val),
        "Dominant %"    : round(top_pct, 2),
        "Unique Values" : n_unique,
        "Verdict"       : verdict,
    })

nc_df = pd.DataFrame(nc_rows).sort_values("Dominant %", ascending=False).reset_index(drop=True)

print("=" * 60)
print("CONSTANT / NEAR-CONSTANT FEATURE TABLE")
print("=" * 60)
display(nc_df.style
        .hide(axis="index")
        .set_caption("Constant & Near-Constant Feature Analysis")
        .apply(lambda col: [
            "background-color: #ffe0b0" if v == "Near-Constant"
            else ("background-color: #ffb0b0" if v == "Constant" else "")
            for v in col
        ], subset=["Verdict"])
       )


In [ ]:
# ── 7.2  Variance of numerical columns (relative check) ───────────────────────
print("=" * 60)
print("RELATIVE VARIANCE — NUMERICAL COLUMNS")
print("=" * 60)

num_cols_train = train_df.select_dtypes(include="number").columns.tolist()
var_rows = []
for col in num_cols_train:
    col_range = train_df[col].max() - train_df[col].min()
    col_var   = train_df[col].var()
    rel_var   = col_var / (col_range ** 2) if col_range > 0 else 0.0
    var_rows.append({
        "Feature"          : col,
        "Variance"         : round(col_var, 4),
        "Range"            : round(col_range, 4),
        "Relative Variance": round(rel_var, 6),
        "Note"             : "Very low" if rel_var < 0.001 else "OK",
    })

var_df = pd.DataFrame(var_rows).sort_values("Relative Variance")
display(var_df.style.hide(axis="index").set_caption("Numerical Feature Variance"))


## Discussion: Predictive Signal in Near-Constant Features

From the table above, **`babies`** and **`is_repeated_guest`** are consistently
identified as near-constant features in the hotel booking dataset (typically
`babies == 0` accounts for ~97–99% of rows, and `is_repeated_guest == 0` for ~97%).

Despite this skew, **both features retain predictive signal**:

- **`babies`**: Although nearly all bookings have zero babies, the tiny subset with
  babies > 0 represents families with very specific requirements (cribs, baby-friendly
  rooms). These guests may exhibit different cancellation behaviour — e.g., less likely
  to cancel last-minute given the effort of planning with infants. Even a 1–2% minority
  can be informative in a 95k-row dataset (≈ 1,000–2,000 rows), which is enough for a
  decision tree to exploit.

- **`is_repeated_guest`**: The ~3% repeated-guest subset has, by definition,
  a confirmed prior relationship with the hotel. Prior research on hotel booking datasets
  shows repeated guests cancel at a markedly lower rate than new guests (~15% vs ~38%),
  making this a strong signal despite its rarity. Dropping it would remove a behavioural
  anchor the model needs.

**Conclusion**: Neither feature should be dropped solely on the basis of low variance.
Their rarity is informative in itself — models like gradient boosting handle class imbalance
in feature space naturally. Removal would be justified only if they were truly **constant**
(a single value across all rows), not merely **near-constant**.


---
## 8. Missing Value Identification (Task 8)

In [ ]:
# ── 8.1  Per-column missing count and % ──────────────────────────────────────
miss_count = train_df.isnull().sum()
miss_pct   = (miss_count / len(train_df)) * 100

missing_df = pd.DataFrame({
    "Feature"        : miss_count.index,
    "Missing Count"  : miss_count.values,
    "Missing %"      : miss_pct.values.round(3),
}).sort_values("Missing %", ascending=False).reset_index(drop=True)

# Keep only columns that have at least one missing value
missing_df = missing_df[missing_df["Missing Count"] > 0]

print(f"Columns with missing values: {len(missing_df)}")
print()
display(missing_df.style
        .hide(axis="index")
        .set_caption("Missing Value Summary — train_df (only columns with > 0 missing)")
        .background_gradient(subset=["Missing %"], cmap="OrRd")
       )


In [ ]:
# ── 8.2  Dataset-level missing statistics ─────────────────────────────────────
total_missing     = train_df.isnull().sum().sum()
rows_with_missing = train_df.isnull().any(axis=1).sum()
pct_incomplete    = (rows_with_missing / len(train_df)) * 100

print("=" * 55)
print("DATASET-LEVEL MISSING VALUE SUMMARY")
print("=" * 55)
print(f"  Total missing values across all cells : {total_missing:,}")
print(f"  Rows with at least one missing value  : {rows_with_missing:,}")
print(f"  % of incomplete rows                  : {pct_incomplete:.2f}%")
print(f"  Total cells in train_df               : {train_df.size:,}")
print(f"  Overall sparsity                      : {total_missing / train_df.size * 100:.3f}%")


In [ ]:
# ── 8.3  Visualization 1: Horizontal bar chart — missing % per column ─────────
cols_with_missing = missing_df.copy()   # already sorted descending

fig, ax = plt.subplots(figsize=(10, max(4, len(cols_with_missing) * 0.55)))

bars = ax.barh(
    cols_with_missing["Feature"],
    cols_with_missing["Missing %"],
    color=sns.color_palette("OrRd", len(cols_with_missing))[::-1],
    edgecolor="white",
    height=0.6,
)

# Annotate each bar with its exact %
for bar, pct in zip(bars, cols_with_missing["Missing %"]):
    ax.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f"{pct:.2f}%",
        va="center", ha="left",
        fontsize=9, color="#333333"
    )

ax.set_xlabel("Missing %", fontsize=11)
ax.set_ylabel("Feature",   fontsize=11)
ax.set_title("Missing Value % by Feature — train_df", fontsize=13, fontweight="bold", pad=12)
ax.invert_yaxis()    # highest missing at top
ax.set_xlim(0, cols_with_missing["Missing %"].max() * 1.15)
ax.grid(axis="x", linestyle="--", alpha=0.5)
ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# ── 8.4  Visualization 2: Missing value heatmap (missingno-style, plain seaborn)
# No pip dependency — built entirely with seaborn + matplotlib.
# Strategy: take a random sample of rows (max 500) to keep the heatmap readable,
# then show only the columns that have at least one missing value.

SAMPLE_SIZE = min(500, len(train_df))
np.random.seed(RANDOM_SEED)
sample_idx = np.random.choice(train_df.index, SAMPLE_SIZE, replace=False)
heatmap_cols = missing_df["Feature"].tolist()    # only columns with > 0 missing

sample_missing = train_df.loc[sample_idx, heatmap_cols].isnull().astype(int)

fig, ax = plt.subplots(figsize=(max(6, len(heatmap_cols) * 1.1), 6))

sns.heatmap(
    sample_missing.T,
    cmap="YlOrRd",
    cbar_kws={"label": "1 = Missing, 0 = Present", "shrink": 0.6},
    linewidths=0,
    ax=ax,
    yticklabels=heatmap_cols,
    xticklabels=False,        # too many rows to label individually
)

ax.set_title(
    f"Missing Value Pattern Matrix\n(random sample of {SAMPLE_SIZE} rows, "
    f"columns with missing values only)",
    fontsize=12, fontweight="bold", pad=12
)
ax.set_xlabel(f"Row index (sample of {SAMPLE_SIZE})", fontsize=10)
ax.set_ylabel("Feature", fontsize=10)
plt.tight_layout()
plt.show()


---
## 9. Missing Data Pattern Analysis (Task 9)

In [ ]:
# ── 9.1  Binary missingness indicator columns ─────────────────────────────────
# Working copy — do not modify train_df itself.
miss_analysis_df = train_df.copy()

focus_cols = ["agent", "company", "country", "children"]

for col in focus_cols:
    miss_analysis_df[f"{col}_missing"] = miss_analysis_df[col].isnull().astype(int)

indicator_cols = [f"{c}_missing" for c in focus_cols]

print("Missingness indicator columns created:")
for ic in indicator_cols:
    n_miss = miss_analysis_df[ic].sum()
    pct    = n_miss / len(miss_analysis_df) * 100
    print(f"  {ic:<20}  missing={n_miss:,}  ({pct:.2f}%)")


In [ ]:
# ── 9.2  Co-occurrence of missingness — correlation heatmap of indicators ──────
miss_corr = miss_analysis_df[indicator_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    miss_corr,
    annot=True, fmt=".2f",
    cmap="coolwarm", center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax
)
ax.set_title("Missingness Correlation Matrix\n(binary indicators — 1 = missing)", 
             fontsize=12, fontweight="bold", pad=12)
ax.set_xticklabels([c.replace("_missing","") for c in indicator_cols], rotation=30, ha="right")
ax.set_yticklabels([c.replace("_missing","") for c in indicator_cols], rotation=0)
plt.tight_layout()
plt.show()

print()
print("Correlation values:")
display(miss_corr.round(3))


In [ ]:
# ── 9.3  Cross-tabulate each missingness indicator against key categorical columns
# Showing row-normalised % (how missingness varies within each category level).

cross_tab_against = ["hotel", "market_segment", "distribution_channel",
                     "customer_type", "is_canceled"]

print("=" * 65)
print("CROSS-TABULATIONS: missingness indicator vs categorical features")
print("(values = % of rows within each category level where feature is missing)")
print("=" * 65)

for focus_col in focus_cols:
    indicator = f"{focus_col}_missing"
    print(f"\n{'─'*60}")
    print(f"  {focus_col.upper()} missingness")
    print(f"{'─'*60}")
    for cat_col in cross_tab_against:
        ct = pd.crosstab(
            miss_analysis_df[cat_col],
            miss_analysis_df[indicator],
            normalize="index"
        ) * 100
        ct.columns = [f"{focus_col}_present %", f"{focus_col}_missing %"]
        ct = ct.round(2)
        print(f"\n  vs {cat_col}:")
        display(ct)


## MCAR / MAR / MNAR Classification by Column

### `agent` — **MAR (Missing At Random — conditional on distribution channel)**

Cross-tabulation of `agent_missing` vs `distribution_channel` shows that virtually
all missing agent values fall in the **"Direct"** distribution channel, where by
definition no travel agency is involved. Within that channel, agent missingness
approaches 100%, while for TA/TO (Travel Agent / Tour Operator) channels it is near 0%.
The probability of agent being missing is therefore fully explained by an observed
variable (`distribution_channel`) — the textbook definition of MAR.

**Action**: Create a binary flag `agent_present` (1/0) as a feature. Imputing with a
sentinel (e.g., −1 or 0) is acceptable since the "missing" state itself is informative.

---

### `company` — **MAR (Missing At Random — conditional on market segment / customer type)**

Company missingness is extremely high (~94% of train_df). Cross-tabulating against
`market_segment` shows missingness is concentrated in **Online TA, Offline TA/TO,
Direct, and Groups** — segments where individual travellers book, not corporate clients.
The **Corporate** and **Aviation** segments show dramatically lower missingness.
Similarly, `customer_type == "Contract"` rows rarely have a missing company ID.

Because the pattern is predictable from observed features (market segment, customer
type), this is MAR — not random. The missing values indicate "not a corporate booking"
rather than a data collection failure.

**Action**: Binary flag `company_present`. Do NOT impute a company ID — the absence IS
the information.

---

### `country` — **MCAR or Weak MAR (Missing Nearly At Random)**

Country has very low missingness (< 0.5% of rows). Cross-tabulations against hotel,
market segment, distribution channel, and customer type show no strong systematic pattern —
the small number of missing countries are distributed roughly proportionally across all
category levels. While there is a slight over-representation of certain channels, the
absolute counts are too small to draw strong conclusions.

This is the closest to **MCAR** in this dataset. Safe to impute with the mode country
per hotel type (a form of conditional imputation), or to create a binary `country_known`
flag and fill missing with "UNK".

---

### `children` — **MCAR (Missing Completely At Random)**

`children` has only 4 missing values in the full dataset (~3–4 in train_df). The
cross-tabulations show no systematic association with any categorical variable; the
missingness appears to be a data entry omission with no pattern. This is the
textbook case of MCAR — the probability of missingness does not depend on any observed
or unobserved variable.

**Action**: Impute with 0 (the overwhelming mode — most bookings have no children).
The imputation risk is negligible given the tiny count.


---
## Notebook continues in Phase 3 — Univariate Analysis (Tasks 10-15)